In [1]:
import pandas as pd
from transformers import T5ForConditionalGeneration, T5Tokenizer,Trainer,TrainingArguments
import os
from datasets import Dataset

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nileshmalode1/samsum-dataset-text-summarization")

print("Path to dataset files:", path)

100%|██████████| 7.99M/7.99M [00:00<00:00, 67.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/nileshmalode1/samsum-dataset-text-summarization/versions/1


In [3]:
print(os.listdir(path))

['samsum_dataset', 'samsum-validation.csv', 'samsum-train.csv', 'samsum-test.csv']


In [4]:
# read dataset
samsum_tarin=pd.read_csv(os.path.join(path, 'samsum-train.csv'))
samsum_validation=pd.read_csv(os.path.join(path, 'samsum-validation.csv'))

In [5]:
print(f"the trainning dataset is :{samsum_tarin.shape}")
print(f"the trainning dataset is :{samsum_validation.shape}")

the trainning dataset is :(14732, 3)
the trainning dataset is :(818, 3)


In [6]:
samsum_tarin.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [7]:
print(samsum_tarin["dialogue"][2])
print("+++++++++++++++")
print(samsum_tarin["summary"][2])

Tim: Hi, what's up?
Kim: Bad mood tbh, I was going to do lots of stuff but ended up procrastinating
Tim: What did you plan on doing?
Kim: Oh you know, uni stuff and unfucking my room
Kim: Maybe tomorrow I'll move my ass and do everything
Kim: We were going to defrost a fridge so instead of shopping I'll eat some defrosted veggies
Tim: For doing stuff I recommend Pomodoro technique where u use breaks for doing chores
Tim: It really helps
Kim: thanks, maybe I'll do that
Tim: I also like using post-its in kaban style
+++++++++++++++
Kim may try the pomodoro technique recommended by Tim to get more stuff done.


In [8]:
# select random state
training_dataset=samsum_tarin.sample(n=4000, random_state=42).reset_index(drop=True)
validation_dataset=samsum_validation.sample(n=400, random_state=42).reset_index(drop=True)

In [9]:
training_dataset

,id,dialogue,summary
0,13811908,Violet: hi! i came across this Austin's articl...,Violet sent Claire Austin's article.
1,13716431,Pat: So does anyone know when the stream is go...,Pat and Lou are waiting for The stream but Kev...
2,13810214,Jane: <gif_file>\r\nJane: Whaddya think? \r\nS...,Jane is updating her Tinder profile tonight an...
3,13729823,"Adam: Do u have a map of Paris?\r\nTom: Yes, W...",Tom has a map of Paris.
4,13681400,"Frank: Hi, how's the family?\r\nMike: great! S...","Mike is happy, because Sam's moved out. Mike a..."
...,...,...,...
3995,13681041,Barry: hello buddy\r\nMichael: hey\r\nBarry: d...,Barry and Michael will watch football instead ...
3996,13818705,Karen: Hey Lisa. Larissa and me have recently ...,Karen and Larissa moved to Belgium and ask Lis...
3997,13821859,"Miles: Hey, guys, I'm so sorry, but I missed t...","Miles has missed the bus, so he may be 15 minu..."
3998,13812716,Emma: did you finish the book I gave you?\r\nL...,"Emma gave ""The First Fifteen Lives of Harry Au..."


In [10]:
training_dataset["dialogue"][2]

"Jane: <gif_file>\r\nJane: Whaddya think? \r\nShona: This ur tinder profile thing?\r\nJane: Yeah, I'm updating my profile tonite. Kinda nervoous though... :( \r\nJane: What if i get another guy like John? o.O\r\nShona: John was a dickhead\r\nJane: preach sistah!\r\nShona: anyhoo - this time I've got u :D No slimeballs for you \r\nJane: Not again *shudders*\r\nJane: You know he forgot my birthday??!!\r\nShona: wanker"

In [11]:
#clean the text
# regular expression
import re
def clean_text(text):
# remove \r\n
  text=re.sub(r"\r\n"," ",text)
  # remove multiple space

  text=re.sub(r"\s+"," ",text)
# remove HTML tags
  text=re.sub(r"<.*?>"," ",text)

  text=text.strip().lower()
  return text

In [12]:
training_dataset["dialogue"]=training_dataset["dialogue"].apply(clean_text)
training_dataset["summary"]=training_dataset["summary"].apply(clean_text)
validation_dataset["summary"]=validation_dataset["summary"].apply(clean_text)
validation_dataset["summary"]=validation_dataset["summary"].apply(clean_text)

In [13]:
training_dataset["dialogue"][2]

"jane:   jane: whaddya think? shona: this ur tinder profile thing? jane: yeah, i'm updating my profile tonite. kinda nervoous though... :( jane: what if i get another guy like john? o.o shona: john was a dickhead jane: preach sistah! shona: anyhoo - this time i've got u :d no slimeballs for you jane: not again *shudders* jane: you know he forgot my birthday??!! shona: wanker"

In [14]:
# load tokenizer
tokenizer=T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [15]:
#  apply tokenizer
def preprocess(samples):
  input=tokenizer(samples["dialogue"],max_length=512,padding="max_length",truncation=True)
  output=tokenizer(samples["summary"],max_length=157,padding="max_length",truncation=True)
  input["labels"]=output["input_ids"]
  return input
training_dataset=training_dataset.apply(preprocess,axis=1)
# [input-id,attension-mask,labels]
validation_dataset=validation_dataset.apply(preprocess ,axis=1)
# [input-id,attension-mask,labels]


In [16]:
model=T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [17]:
training_args = TrainingArguments(
  num_train_epochs=5,
  learning_rate=5e-5,
  warmup_steps=500,
  weight_decay=0.01,
  per_device_train_batch_size=4,
  per_device_eval_batch_size=4,
  gradient_accumulation_steps=4,
  do_eval=True,
  eval_steps=500
)
trainer=Trainer(
    model=model,
    args= training_args,
    train_dataset=training_dataset,

    eval_dataset=validation_dataset,
)

In [18]:
trainer.train()

Step,Training Loss
500,13.834073
1000,1.454467


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=6.395006396484375, metrics={'train_runtime': 1112.8939, 'train_samples_per_second': 17.971, 'train_steps_per_second': 1.123, 'total_flos': 2706836029440000.0, 'train_loss': 6.395006396484375, 'epoch': 5.0})

In [20]:
model.save_pretrained("/content/Saved_Files/Summary_Model")
tokenizer.save_pretrained("/content/Saved_Files/Tokenizer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/Saved_Files/Tokenizer/tokenizer_config.json',
 '/content/Saved_Files/Tokenizer/tokenizer.json')

In [21]:
import re
def clean(text):

# remove \r\n
  text=re.sub(r"\r\n"," ",text)
  # remove multiple space

  text=re.sub(r"\s+"," ",text)
# remove HTML tags
  text=re.sub(r"<.*?>"," ",text)

  text=text.strip().lower()
  return text

In [26]:
import torch

def summarization(dialogue):
  dialogue=clean(dialogue)
  input_ids=tokenizer.encode(dialogue,return_tensors="pt",max_length=512,truncation=True)
  # Move input_ids to the same device as the model
  input_ids = input_ids.to(model.device)
  outputs=model.generate(
      input_ids=input_ids,
      max_length=157,
      min_length=40,
      length_penalty=2.0,
      num_beams=4,
      early_stopping=True
  )
  summary=tokenizer.decode(outputs[0],skip_special_tokens=True)
  return summary

In [27]:
# test sample
detailed_dialogue="""
"Lenny: Babe, can you help me with something?
Bob: Sure, what's up?
Lenny: Which one should I pick?
Bob: Send me photos
Lenny:  <file_photo>
Lenny:  <file_photo>
Lenny:  <file_photo>
Bob: I like the first ones best
Lenny: But I already have purple trousers. Does it make sense to have two pairs?
Bob: I have four black pairs :D :D
Lenny: yeah, but shouldn't I pick a different color?
Bob: what matters is what you'll give you the most outfit options
Lenny: So I guess I'll buy the first or the third pair then
Bob: Pick the best quality then
Lenny: ur right, thx
Bob: no prob

"""
summary=summarization(detailed_dialogue)
print(summary)

bob has four black pairs. lenny will buy the first or the third pair then lenny will pick the best color. bob will buy the first or the third pair.


In [29]:
import shutil
shutil.make_archive("summarization_model", 'zip', "/content/Saved_Files")
from google.colab import files
files.download("summarization_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>